# Run this project in Google Colab

1. Choose **File > Save a copy in Drive** to keep your own notebook.
2. Use a normal CPU runtime. A GPU is not needed.
3. Choose **Runtime > Run all**. The first code cell downloads the dataset and installs the required packages.
4. If Colab asks for a session restart after installation, restart it and choose **Runtime > Run all** again.
5. When execution finishes, choose **File > Download > Download .ipynb** to save your notebook with its outputs.

No separate data upload is needed. The data comes from the project repository at a fixed revision.
This Colab copy starts with cleared outputs so that the displayed results come from your own run.
The observations describe the earlier verified results; compare them with the results from your run.
The live app and GitHub workflow were verified separately on 12 October 2026.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import subprocess
import sys

# Download the course data and real deployment evidence from my repository.
revision = "d09282c14891074526755f7122c16914a0114652"
base_url = f"https://raw.githubusercontent.com/pramod-collab/tourism-purchase-prediction/{revision}/"
project_files = [
    "tourism_project/data/tourism.csv",
    "tourism_project/requirements.txt",
    "tourism_project/deployment/requirements.txt",
    "deployment_evidence.json",
    "evidence/github_structure.jpg",
    "evidence/github_workflow.jpg",
    "evidence/streamlit_app.jpg",
    "evidence/streamlit_form.jpg",
]
for name in project_files:
    destination = Path(name)
    destination.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(base_url + name, destination)

# Install the same package versions used for the submitted project.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                "tourism_project/requirements.txt"], check=True)
print("Setup completed. The dataset and deployment screenshots are ready.")
print("If Colab asks to restart the session, restart it and then run all cells again.")


### Observations

This cell prepares the Colab environment. It downloads the same dataset and real screenshots used in the project and installs the required libraries. The analysis starts below.

# Tourism package purchase prediction

## Business context

Visit with Us wants to find customers who may buy a tourism package. A useful contact list
can help the sales team spend less time on customers who are unlikely to buy.

## Objective

Build and test a purchase prediction model, save it, create a Streamlit app, and prepare a
GitHub Actions workflow that repeats data preparation and model training when the project changes.
The prediction is intended to be made **before the sales call**.

The supplied target, `ProdTaken`, describes historical package purchases. It does not identify
Wellness purchases separately. I therefore treat this as a model of general purchase interest.
Its use for the new Wellness package needs to be checked in a pilot campaign.

## Submission and deployment status

This notebook follows the supplied rubric and the learner template. Both use Streamlit
Community Cloud, so that is the deployment platform used here. The older business-context
document mentions Hugging Face; I have followed the rubric where they differ.

The analysis, training and app checks were executed locally. The GitHub Actions pipeline
also completed successfully and committed the trained model back to main. The public
Streamlit app was tested with sample inputs on 12 October 2026. Real links and screenshots
are included in the Output Evaluation section.

## How to run

Use Python 3.13 and install the packages in `tourism_project/requirements.txt` before running
all cells in order. Keep the supplied CSV in `tourism_project/data/` or beside this notebook
as `tourism.csv`. Run the notebook from the extracted project folder. Code cells that write
files create the same source files used by GitHub Actions. Every code cell is followed by
an observations cell.

## 1. Setup and data registration

In [ ]:
from pathlib import Path
import json
import shutil
import sys
import platform
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
for folder in ['tourism_project/data', 'tourism_project/model_building',
               'tourism_project/deployment', 'tourism_project/reports',
               'tourism_project/splits', '.github/workflows', 'tests', 'evidence']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
for name in ['tourism_project/__init__.py', 'tourism_project/model_building/__init__.py']:
    (ROOT / name).touch()
source = ROOT / 'tourism.csv'
data_path = ROOT / 'tourism_project/data/tourism.csv'
if not data_path.exists():
    if not source.exists():
        raise FileNotFoundError('Place tourism.csv beside this notebook before running.')
    shutil.copy2(source, data_path)
pd.set_option('display.max_columns', 25)
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False,
                     'axes.spines.right': False})
print('Python:', platform.python_version())
print('Dataset found:', data_path.exists())
print('Project folders are ready.')

### Observations

The master folder contains separate data, training, deployment and report folders.
The supplied dataset is copied without changing its contents. The same folder paths are
used in the notebook and the automated workflow.

In [ ]:
%%writefile tourism_project/model_building/common.py
from pathlib import Path
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
PROJECT = ROOT / 'tourism_project'
DATA = PROJECT / 'data'
SPLITS = PROJECT / 'splits'
REPORTS = PROJECT / 'reports'
DEPLOY = PROJECT / 'deployment'
TARGET = 'ProdTaken'
SEED = 42
EXPECTED = ['CustomerID', 'ProdTaken', 'Age', 'TypeofContact', 'CityTier',
            'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting',
            'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar',
            'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore',
            'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome']
# These fields describe the current contact or events after contact.
AFTER_CONTACT = ['TypeofContact', 'DurationOfPitch', 'NumberOfFollowups',
                 'ProductPitched', 'PitchSatisfactionScore']
CATEGORICAL = ['CityTier', 'Occupation', 'Gender', 'MaritalStatus', 'Designation']
NUMERIC = ['Age', 'NumberOfPersonVisiting', 'PreferredPropertyStar',
           'NumberOfTrips', 'Passport', 'OwnCar', 'NumberOfChildrenVisiting',
           'MonthlyIncome']
FEATURES = NUMERIC + CATEGORICAL

def clean_data(raw):
    data = raw.copy()
    for col in data.select_dtypes('object'):
        data[col] = data[col].str.strip()
    data['Gender'] = data['Gender'].replace({'Fe Male': 'Female'})
    # Keep Single and Unmarried separate: the dictionary does not say they are identical.
    return data[FEATURES + [TARGET]]

def profile_groups(features):
    # Identical available profiles stay in the same partition, even with different IDs.
    return pd.util.hash_pandas_object(features, index=False).astype(str)

### Observations

This file keeps column names and paths in one place.
CustomerID and the exported row number are excluded. Contact type, pitch duration, follow-ups,
product pitched and pitch satisfaction are excluded because this model is for use before contact.
The remaining 13 features are assumed to be available in an existing customer record.
If travel preferences are collected only during the call, they must also be excluded before real use.

In [ ]:
%%writefile tourism_project/model_building/data_register.py
import hashlib
import json
import pandas as pd
from tourism_project.model_building.common import DATA, REPORTS, EXPECTED

def register():
    path = DATA / 'tourism.csv'
    df = pd.read_csv(path)
    missing = sorted(set(EXPECTED) - set(df.columns))
    extra = [c for c in df.columns if c not in EXPECTED and not c.startswith('Unnamed:')]
    if missing or extra:
        raise ValueError(f'Invalid schema. Missing: {missing}; unexpected: {extra}')
    if df.empty or df.CustomerID.isna().any() or df.CustomerID.duplicated().any():
        raise ValueError('The file must contain rows and unique, nonmissing customer IDs.')
    if df.ProdTaken.isna().any() or set(df.ProdTaken.unique()) != {0, 1}:
        raise ValueError('ProdTaken must contain both 0 and 1, without missing labels.')
    numeric = [c for c in EXPECTED if c not in
               ['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
                'MaritalStatus', 'Designation']]
    for col in numeric:
        pd.to_numeric(df[col], errors='raise')
    for col in ['Passport', 'OwnCar']:
        if not df[col].dropna().isin([0, 1]).all():
            raise ValueError(f'{col} must be binary.')
    summary = {'rows': len(df), 'columns': len(df.columns),
               'buyers': int(df.ProdTaken.sum()),
               'buyer_rate': float(df.ProdTaken.mean()),
               'missing_values': int(df.isna().sum().sum()),
               'duplicate_rows': int(df.duplicated().sum()),
               'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    REPORTS.mkdir(parents=True, exist_ok=True)
    (REPORTS / 'data_registration.json').write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    return summary

if __name__ == '__main__':
    register()

### Observations

The validation script checks the column names,
customer IDs, target labels and numeric fields. It also saves a file fingerprint, so a later
run can tell whether the dataset changed. A bad schema stops the pipeline early.

In [ ]:
from tourism_project.model_building.data_register import register
registration = register()

### Observations

The file has 4,128 rows and 21 columns, including an exported index.
There are 797 buyers and 3,331 non-buyers. The buyer rate is 19.31%. There are no missing
values, duplicate full rows or duplicate customer IDs in this copy.

## 2. Understanding the data

In [ ]:
raw = pd.read_csv(data_path)
display(raw.head())
display(pd.DataFrame({'data_type': raw.dtypes.astype(str),
                      'missing': raw.isna().sum(),
                      'unique_values': raw.nunique()}))
display(raw.describe().round(2))

### Observations

`ProdTaken` is the target: 1 means purchased and 0 means did not purchase.
The data contains customer details and sales interaction details. Ages range from 18 to 61.
Some numeric columns contain whole numbers stored as decimals; that does not change their meaning.
The source file has already been supplied without missing values, so no rows need to be
removed for missing data. Imputation is still included for future inputs.

In [ ]:
for column in raw.select_dtypes('object'):
    print(column, raw[column].value_counts().to_dict())
print('Children greater than party size:',
      int((raw.NumberOfChildrenVisiting > raw.NumberOfPersonVisiting).sum()))
print('Designation and pitched product:')
display(pd.crosstab(raw.Designation, raw.ProductPitched))

### Observations

`Fe Male` is a spelling variation of `Female` and is corrected.
`Single` and `Unmarried` are kept separate because the dictionary does not explain whether
they mean the same thing. There are only two Free Lancer records, so conclusions for that
group would be unreliable. No row has more children than people travelling.

Designation corresponds exactly to the pitched product in this dataset. Designation is retained
as a customer attribute only under the assumption that it was recorded independently before
the pitch. This relationship needs checking in the original data collection process.

## 3. Data preparation and train-test split

In [ ]:
%%writefile tourism_project/model_building/prep.py
import json
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from tourism_project.model_building.common import (
    DATA, SPLITS, REPORTS, TARGET, SEED, clean_data, profile_groups)

def prepare():
    raw = pd.read_csv(DATA / 'tourism.csv')
    data = clean_data(raw)
    X, y = data.drop(columns=TARGET), data[TARGET].astype(int)
    groups = profile_groups(X)
    # One of five folds is held out. Grouping prevents identical profiles leaking.
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_idx, test_idx = next(splitter.split(X, y, groups))
    SPLITS.mkdir(parents=True, exist_ok=True)
    for name, frame in [('Xtrain', X.iloc[train_idx]), ('Xtest', X.iloc[test_idx]),
                        ('ytrain', y.iloc[train_idx]), ('ytest', y.iloc[test_idx])]:
        frame.to_csv(SPLITS / f'{name}.csv', index=False)
    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])
    assert not train_groups.intersection(test_groups)
    summary = {'train_rows': len(train_idx), 'test_rows': len(test_idx),
               'features': list(X.columns), 'train_buyer_rate': float(y.iloc[train_idx].mean()),
               'test_buyer_rate': float(y.iloc[test_idx].mean()),
               'repeated_profile_rows': int(X.duplicated().sum()),
               'shared_profiles': 0, 'random_state': SEED}
    (REPORTS / 'split_summary.json').write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    return summary

if __name__ == '__main__':
    prepare()

### Observations

Cleaning fixes spelling and removes unavailable or unnecessary columns.
There are 194 repeated rows of model input values, although customer IDs are different.
These records are retained, but matching profiles stay in the same split. The first of five
stratified group folds is the holdout set. The split is close to 80:20; grouping can make
the sizes and buyer rates slightly different.

In [ ]:
from tourism_project.model_building.prep import prepare
split_summary = prepare()
Xtrain = pd.read_csv('tourism_project/splits/Xtrain.csv')
Xtest = pd.read_csv('tourism_project/splits/Xtest.csv')
ytrain = pd.read_csv('tourism_project/splits/ytrain.csv').iloc[:, 0]
ytest = pd.read_csv('tourism_project/splits/ytest.csv').iloc[:, 0]
train_data = Xtrain.assign(ProdTaken=ytrain)
display(pd.DataFrame({'rows': [len(Xtrain), len(Xtest)],
                      'buyer_rate': [ytrain.mean(), ytest.mean()]},
                     index=['Training', 'Test']))

### Observations

The training set has 3,300 rows and the test set has 828 rows. Their buyer rates are 19.18% and 19.81%. There are no identical cleaned profiles shared between the two sets. Four CSV files are saved for the training job. Preprocessing is fitted later, using training data only.

## 4. Exploratory analysis on the training set

Detailed plots use the training set. The test set is reserved for the final model evaluation.
The relationships below are associations in historical data, not proof that a feature causes a purchase.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
train_data.ProdTaken.value_counts().sort_index().plot.bar(ax=axes[0], color=['#6b8292', '#25785c'])
axes[0].set(title='Purchase counts', xlabel='Purchased (0 = No, 1 = Yes)', ylabel='Customers')
axes[0].tick_params(axis='x', rotation=0)
train_data.Age.hist(ax=axes[1], bins=15, color='#6b8292', edgecolor='white')
axes[1].set(title='Age distribution', xlabel='Age', ylabel='Customers')
train_data.MonthlyIncome.hist(ax=axes[2], bins=25, color='#6b8292', edgecolor='white')
axes[2].set(title='Monthly income distribution', xlabel='Monthly income', ylabel='Customers')
plt.tight_layout()
plt.show()
display(train_data[['Age', 'MonthlyIncome', 'NumberOfTrips']].describe().round(2))

### Observations

Non-buyers are the larger group. A model that predicts every customer as a non-buyer
can look good on accuracy while finding no buyers. Monthly income has a long upper tail.
I keep unusually high incomes and trip counts because there is no supplied rule showing
that these values are errors. Tree models do not require standard scaling.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, column in zip(axes, ['Passport', 'Designation', 'MaritalStatus']):
    summary = train_data.groupby(column).ProdTaken.agg(['count', 'mean'])
    display(summary.rename(columns={'mean': 'purchase_rate'}).round(3))
    (summary['mean'] * 100).plot.bar(ax=ax, color='#25785c')
    ax.set(title=f'Purchase rate by {column}', ylabel='Purchase rate (%)')
    ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()
print('Purchase rate by age band:')
display(train_data.groupby(pd.cut(train_data.Age, [17, 30, 40, 50, 65]),
                          observed=True).ProdTaken.agg(['count', 'mean']).round(3))

### Observations

In the training data, the purchase rate is 35.1% for passport holders and 12.4% for customers without a passport. The designation, marital-status and age tables show further differences in historical purchase rates. These comparisons help describe the data, but they do not establish causes or guarantee future purchases. Group sizes should be considered alongside the rates.

## 5. Model building and experiment tracking

I compare Decision Tree, Random Forest and Gradient Boosting. A majority-class model is the
baseline. All model selection uses three training folds, with identical profiles kept together.
The pipeline learns imputation and one-hot encoding separately inside each fold.

**Selection metric:** average precision, which checks how well buyers are ranked across
different score thresholds. I also report precision, recall, F1, F2, accuracy and ROC AUC.
Recall is the share of actual buyers found. Precision is the share of predicted buyers who
actually bought. F2 gives recall more weight than precision.

The model family and parameters are chosen by cross-validation average precision. The contact
threshold is then chosen by F2 on out-of-fold training predictions. This is a practical choice
for finding buyers when no contact cost or capacity has been supplied. It is not a profit optimum.
Reusing training folds for tuning can make their scores optimistic; the untouched holdout is
used for the final check. MLflow records all 28 parameter combinations.

In [ ]:
%%writefile tourism_project/model_building/train.py
import json
import platform
import hashlib
import joblib
import mlflow
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tourism_project.model_building.common import (
    ROOT, DATA, SPLITS, REPORTS, DEPLOY, NUMERIC, CATEGORICAL, SEED, profile_groups)

def evaluate(y, probability, threshold):
    pred = (probability >= threshold).astype(int)
    return {'accuracy': float(accuracy_score(y, pred)),
            'precision': float(precision_score(y, pred, zero_division=0)),
            'recall': float(recall_score(y, pred)),
            'f1': float(f1_score(y, pred, zero_division=0)),
            'f2': float(fbeta_score(y, pred, beta=2, zero_division=0)),
            'roc_auc': float(roc_auc_score(y, probability)),
            'average_precision': float(average_precision_score(y, probability))}

def train():
    Xtrain = pd.read_csv(SPLITS / 'Xtrain.csv')
    ytrain = pd.read_csv(SPLITS / 'ytrain.csv').iloc[:, 0]
    Xtest = pd.read_csv(SPLITS / 'Xtest.csv')
    ytest = pd.read_csv(SPLITS / 'ytest.csv').iloc[:, 0]
    preprocessor = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), NUMERIC),
        ('category', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), CATEGORICAL)
    ])
    # Fit imputation and encoding inside each fold, not before cross-validation.
    from sklearn.tree import DecisionTreeClassifier
    candidates = {
        'Decision Tree': (DecisionTreeClassifier(random_state=SEED), {
            'model__max_depth': [4, 8, None],
            'model__min_samples_leaf': [2, 10],
            'model__class_weight': [None, 'balanced']}),
        'Random Forest': (RandomForestClassifier(random_state=SEED, n_jobs=1), {
            'model__n_estimators': [150], 'model__max_depth': [10, None],
            'model__min_samples_leaf': [1, 3],
            'model__class_weight': [None, 'balanced']}),
        'Gradient Boosting': (GradientBoostingClassifier(random_state=SEED), {
            'model__n_estimators': [100, 200],
            'model__learning_rate': [0.05, 0.1], 'model__max_depth': [2, 3]})
    }
    groups = profile_groups(Xtrain)
    folds = list(StratifiedGroupKFold(n_splits=3, shuffle=True,
                 random_state=SEED).split(Xtrain, ytrain, groups))
    tracking = ROOT / 'mlruns'
    mlflow.set_tracking_uri(tracking.as_uri())
    mlflow.set_experiment('Tourism pre-contact purchase prediction')
    REPORTS.mkdir(parents=True, exist_ok=True)
    DEPLOY.mkdir(parents=True, exist_ok=True)
    rows, best_score, best_search, best_name = [], -1, None, None
    with mlflow.start_run(run_name='model_comparison') as parent:
        mlflow.log_params({'seed': SEED, 'cv_folds': 3, 'selection_metric': 'average_precision',
                           'feature_count': Xtrain.shape[1], 'grouped_cv': True})
        for name, (estimator, grid) in candidates.items():
            pipeline = Pipeline([('preprocess', clone(preprocessor)), ('model', estimator)])
            search = GridSearchCV(pipeline, grid, scoring='average_precision',
                                  cv=folds, n_jobs=1, return_train_score=True, error_score='raise')
            search.fit(Xtrain, ytrain)
            results = search.cv_results_
            for i, params in enumerate(results['params']):
                row = {'model': name, 'parameters': json.dumps(params, sort_keys=True),
                       'cv_average_precision': float(results['mean_test_score'][i]),
                       'cv_std': float(results['std_test_score'][i]),
                       'train_average_precision': float(results['mean_train_score'][i])}
                rows.append(row)
                with mlflow.start_run(run_name=f'{name}_{i+1}', nested=True):
                    mlflow.log_param('model_family', name)
                    mlflow.log_params(params)
                    mlflow.log_metrics({k: row[k] for k in
                        ['cv_average_precision', 'cv_std', 'train_average_precision']})
            print(f'{name}: best cross-validation AP = {search.best_score_:.4f}', flush=True)
            if search.best_score_ > best_score:
                best_score, best_search, best_name = search.best_score_, search, name
        comparison = pd.DataFrame(rows).sort_values('cv_average_precision', ascending=False)
        comparison.to_csv(REPORTS / 'experiments.csv', index=False)
        best = best_search.best_estimator_
        # These predictions come only from training folds. The test set is not used.
        oof = cross_val_predict(clone(best), Xtrain, ytrain, cv=folds,
                               method='predict_proba', n_jobs=1)[:, 1]
        thresholds = np.round(np.arange(0.10, 0.81, 0.05), 2)
        scores = [evaluate(ytrain, oof, t) for t in thresholds]
        threshold_table = pd.DataFrame(scores)
        threshold_table.insert(0, 'threshold', thresholds)
        # F2 weights recall more than precision. Ties use the higher threshold.
        choice = threshold_table.sort_values(['f2', 'threshold'], ascending=False).iloc[0]
        threshold = float(choice['threshold'])
        threshold_table.to_csv(REPORTS / 'thresholds.csv', index=False)
        # The final model already fits all training rows. Holdout remains untouched.
        probability = best.predict_proba(Xtest)[:, 1]
        pred = (probability >= threshold).astype(int)
        metrics = evaluate(ytest, probability, threshold)
        default_metrics = evaluate(ytest, probability, 0.5)
        dummy = DummyClassifier(strategy='prior').fit(Xtrain, ytrain)
        baseline = evaluate(ytest, dummy.predict_proba(Xtest)[:, 1], 0.5)
        joblib.dump(best, DEPLOY / 'model.joblib', compress=3)
        metadata = {'model_family': best_name, 'threshold': threshold,
            'features': list(Xtrain.columns), 'best_parameters': best_search.best_params_,
            'best_cv_average_precision': best_score,
            'test_metrics': metrics, 'test_metrics_at_0_5': default_metrics,
            'baseline_metrics': baseline, 'confusion_matrix': confusion_matrix(ytest, pred).tolist(),
            'train_metrics': evaluate(ytrain, best.predict_proba(Xtrain)[:, 1], threshold),
            'training_rows': len(Xtrain), 'test_rows': len(Xtest),
            'python': platform.python_version(), 'sklearn': sklearn.__version__,
            'data_sha256': hashlib.sha256((DATA / 'tourism.csv').read_bytes()).hexdigest(),
            'mlflow_parent_run_id': parent.info.run_id}
        (DEPLOY / 'metadata.json').write_text(json.dumps(metadata, indent=2))
        (REPORTS / 'metrics.json').write_text(json.dumps(metadata, indent=2))
        pd.DataFrame({'actual': ytest, 'probability': probability, 'prediction': pred}).to_csv(
            REPORTS / 'test_predictions.csv', index=False)
        (REPORTS / 'classification_report.txt').write_text(
            classification_report(ytest, pred, target_names=['Not purchased', 'Purchased'], zero_division=0))
        mlflow.log_params({'selected_model': best_name, 'selected_threshold': threshold,
                          **best_search.best_params_})
        mlflow.log_metrics({f'test_{k}': v for k, v in metrics.items()})
        mlflow.log_artifact(str(DEPLOY / 'model.joblib'), artifact_path='model')
        mlflow.log_artifact(str(DEPLOY / 'metadata.json'), artifact_path='model')
        mlflow.log_artifact(str(REPORTS / 'experiments.csv'))
        mlflow.log_artifact(str(REPORTS / 'thresholds.csv'))
    print('Selected model:', best_name)
    print('Threshold selected on training folds:', threshold)
    print(pd.DataFrame({'Baseline': baseline, 'Model at 0.5': default_metrics,
                        'Selected threshold': metrics}).round(4).to_string())
    print('Confusion matrix:', metadata['confusion_matrix'])
    print('MLflow parent run:', metadata['mlflow_parent_run_id'])
    return metadata

if __name__ == '__main__':
    train()

### Observations

The training script compares the three model families and logs every
parameter combination as a child MLflow run. The parent run records the final choice,
test metrics and model files. Both preprocessing and the classifier are saved together,
so the app uses the same transformations as training. The test set is not used to choose
parameters or the threshold.

In [ ]:
from tourism_project.model_building.train import train
results = train()

### Observations

The selected model is Random Forest. The best average-precision scores from cross-validation are Random Forest: 0.711; Gradient Boosting: 0.654; Decision Tree: 0.535. The model was selected using training folds, before looking at test performance. The saved model contains both preprocessing and prediction steps. Its parameters, metrics and artifacts have been recorded in MLflow.

In [ ]:
experiments = pd.read_csv('tourism_project/reports/experiments.csv')
best_by_family = experiments.sort_values('cv_average_precision', ascending=False).groupby('model').head(1)
display(best_by_family[['model', 'cv_average_precision', 'cv_std', 'train_average_precision', 'parameters']])
print('Number of parameter combinations:', len(experiments))
import mlflow
mlflow.set_tracking_uri((ROOT / 'mlruns').as_uri())
experiment = mlflow.get_experiment_by_name('Tourism pre-contact purchase prediction')
runs = mlflow.search_runs([experiment.experiment_id])
parent_id = results['mlflow_parent_run_id']
current_runs = runs[(runs.run_id == parent_id) | (runs['tags.mlflow.parentRunId'] == parent_id)]
display(current_runs[['run_id', 'status', 'tags.mlflow.runName', 'metrics.cv_average_precision']].head(12))
print('Recorded runs for this execution:', len(current_runs))

### Observations

The table compares the best setting from each model family. The training score
can be higher than the cross-validation score because the model has already seen the
training examples. MLflow contains 28 child runs and one parent run for this execution.
The CSV report makes the same parameter results easy to review without running a server.

The selected Random Forest has training average precision of 1.000, compared
with about 0.711 in cross-validation and 0.739 on the test set. This gap shows
overfitting. It still ranks unseen buyers better than the other tested models,
but its training score should not be used as the expected future result.

In [ ]:
threshold_table = pd.read_csv('tourism_project/reports/thresholds.csv')
display(threshold_table[['threshold', 'precision', 'recall', 'f1', 'f2']].round(4))
ax = threshold_table.plot(x='threshold', y=['precision', 'recall', 'f2'], marker='o')
ax.axvline(results['threshold'], color='black', linestyle='--', label='Selected threshold')
ax.set(title='Threshold choice using training-fold predictions', ylabel='Score', ylim=(0, 1.05))
ax.legend()
plt.tight_layout()
plt.show()

### Observations

The selected threshold is 0.20. On out-of-fold training predictions, it gives precision 48.1%, recall 86.4% and F2 0.745. Lower thresholds usually find more buyers but also add more non-buyers to the contact list. This threshold favours recall and is provisional because contact costs and capacity were not supplied.

## 6. Final test evaluation

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
predictions = pd.read_csv('tourism_project/reports/test_predictions.csv')
metrics_table = pd.DataFrame({'Majority baseline': results['baseline_metrics'],
                              'Model at 0.5': results['test_metrics_at_0_5'],
                              'Selected threshold': results['test_metrics']})
display(metrics_table.round(4))
print(Path('tourism_project/reports/classification_report.txt').read_text())
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ConfusionMatrixDisplay.from_predictions(predictions.actual, predictions.prediction,
    display_labels=['No purchase', 'Purchase'], ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Test confusion matrix')
PrecisionRecallDisplay.from_predictions(predictions.actual, predictions.probability, ax=axes[1])
axes[1].axhline(predictions.actual.mean(), linestyle='--', color='grey', label='Buyer rate')
axes[1].legend()
axes[1].set_title('Test precision-recall curve')
RocCurveDisplay.from_predictions(predictions.actual, predictions.probability, ax=axes[2])
axes[2].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[2].set_title('Test ROC curve')
plt.tight_layout()
plt.show()

### Observations

On 828 held-out customers, the model correctly identifies 143 of the 164 buyers and misses 21. It also flags 118 non-buyers and correctly leaves 546 non-buyers unflagged. Precision is 54.8%, recall is 87.2%, F1 is 0.673, and average precision is 0.739.

The majority baseline has accuracy 80.2%, but zero recall for buyers. That explains why accuracy alone is not enough. The test set was not used to select the model or threshold. These are results for historical data, not a measured increase in campaign sales.

In [ ]:
import joblib
from sklearn.inspection import permutation_importance
saved_model = joblib.load('tourism_project/deployment/model.joblib')
importance = permutation_importance(saved_model, Xtest, ytest, scoring='average_precision',
                                    n_repeats=5, random_state=42, n_jobs=1)
importance_table = pd.DataFrame({'feature': Xtest.columns,
                                'AP decrease': importance.importances_mean,
                                'repeat_std': importance.importances_std}).sort_values('AP decrease', ascending=False)
display(importance_table.round(4))
importance_table.sort_values('AP decrease').plot.barh(x='feature', y='AP decrease', legend=False, color='#6b8292')
plt.title('Permutation importance on the test set')
plt.xlabel('Decrease in average precision when shuffled')
plt.tight_layout()
plt.show()
importance_table.to_csv('tourism_project/reports/feature_importance.csv', index=False)

### Observations

The largest average-precision decreases occur when Passport, Designation, CityTier are shuffled. This suggests the model relies on these variables for ranking customers. Permutation importance measures model reliance, not causation. Correlated features can share importance, and the repeated-shuffle standard deviation is not a population confidence interval. This test-set analysis is descriptive; it was not used to change the model.

In [ ]:
audit = Xtest[['Gender']].copy()
audit['actual'] = ytest.to_numpy()
audit['predicted'] = predictions.prediction.to_numpy()
group_rows = []
for group, frame in audit.groupby('Gender'):
    positives = frame.actual == 1
    negatives = frame.actual == 0
    group_rows.append({'Gender': group, 'customers': len(frame), 'actual_buyers': int(positives.sum()),
                      'recall': frame.loc[positives, 'predicted'].mean(),
                      'false_positive_rate': frame.loc[negatives, 'predicted'].mean()})
display(pd.DataFrame(group_rows).round(3))

### Observations

This small check compares errors by gender. Different error rates can affect who
receives a marketing call. These results do not establish fairness, and small subgroups may
be noisy. Before real use, compare a model without gender and marital status and review
other relevant groups. The score should help prioritize calls, not deny service.

## 7. Streamlit deployment files

In [ ]:
%%writefile tourism_project/deployment/app.py
from pathlib import Path
import json
import joblib
import pandas as pd
import streamlit as st

HERE = Path(__file__).resolve().parent
st.set_page_config(page_title='Tourism purchase prediction', layout='centered')

@st.cache_resource
def load_model():
    return joblib.load(HERE / 'model.joblib'), json.loads((HERE / 'metadata.json').read_text())

model, metadata = load_model()
st.title('Tourism purchase prediction')
st.write('Enter customer details available before the sales call.')
st.caption('Course project for Visit with Us. The estimate is based on past package purchases.')

with st.form('customer'):
    left, right = st.columns(2)
    with left:
        age = st.number_input('Age', 18, 100, 35)
        income = st.number_input('Monthly income', 0, 1000000, 23000, step=500)
        occupation = st.selectbox('Occupation', ['Salaried', 'Small Business', 'Large Business', 'Free Lancer'])
        designation = st.selectbox('Designation', ['Executive', 'Manager', 'Senior Manager', 'AVP', 'VP'])
        gender = st.selectbox('Gender', ['Female', 'Male'])
        marital = st.selectbox('Marital status', ['Married', 'Single', 'Unmarried', 'Divorced'])
    with right:
        tier = st.selectbox('City tier', [1, 2, 3])
        people = st.number_input('People travelling', 1, 20, 3)
        children = st.number_input('Children under 5', 0, 20, 1)
        stars = st.selectbox('Preferred hotel stars', [3, 4, 5])
        trips = st.number_input('Trips per year', 0, 50, 3)
        passport = st.selectbox('Valid passport', ['No', 'Yes'])
        car = st.selectbox('Owns a car', ['No', 'Yes'])
    submitted = st.form_submit_button('Predict purchase')

if submitted:
    if children > people:
        st.error('Children cannot exceed the total number of people travelling.')
    else:
        customer = pd.DataFrame([{
            'Age': age, 'NumberOfPersonVisiting': people, 'PreferredPropertyStar': stars,
            'NumberOfTrips': trips, 'Passport': int(passport == 'Yes'),
            'OwnCar': int(car == 'Yes'), 'NumberOfChildrenVisiting': children,
            'MonthlyIncome': income, 'CityTier': tier, 'Occupation': occupation,
            'Gender': gender, 'MaritalStatus': marital, 'Designation': designation
        }])[metadata['features']]
        probability = float(model.predict_proba(customer)[0, 1])
        st.metric('Estimated purchase likelihood', f'{probability:.1%}')
        if probability >= metadata['threshold']:
            st.success('Suggested action: include this customer in the contact list.')
        else:
            st.info('Suggested action: lower priority for this campaign.')
        st.caption('This is a model estimate, not a confirmed purchase or a calibrated probability.')
        if age > 61 or people > 5 or trips > 22 or income < 1000 or income > 98678:
            st.warning('Some inputs are outside the ranges seen in the training data. Interpret carefully.')
        with st.expander('Entered customer details'):
            st.dataframe(customer, hide_index=True)
st.caption('Use only information already held before contact. Do not use the score to deny service.')

### Observations

The app loads the saved model and threshold, collects customer inputs into one
dataframe, and shows a purchase estimate and contact suggestion. It does not ask for pitch
information. It rejects impossible party sizes and warns about values outside the observed
ranges. The displayed probability is not calibrated, so it should be treated as an estimate.

In [ ]:
%%writefile tourism_project/deployment/requirements.txt
numpy==2.3.5
pandas==2.3.3
scikit-learn==1.7.2
scipy==1.16.3
joblib==1.5.2
streamlit==1.51.0

### Observations

The app dependencies are pinned to the versions used for local training.
Using matching versions reduces problems when loading the saved scikit-learn model.

In [ ]:
%%writefile tourism_project/requirements.txt
-r deployment/requirements.txt
mlflow-skinny==3.0.1
matplotlib==3.10.6
nbformat==5.10.4
nbconvert==7.16.6
nbclient==0.10.2
ipykernel==6.30.1
pytest==8.4.2
PyYAML==6.0.2

### Observations

This dependency file adds tracking, plotting, notebook and testing packages.
It includes the deployment dependency file so the training environment and app use the
same model-library versions.

In [ ]:
%%writefile tests/test_project.py
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from streamlit.testing.v1 import AppTest
from tourism_project.model_building.common import ROOT, SPLITS, DEPLOY, profile_groups

def test_no_profile_leakage():
    train = pd.read_csv(SPLITS / 'Xtrain.csv')
    test = pd.read_csv(SPLITS / 'Xtest.csv')
    assert not set(profile_groups(train)) & set(profile_groups(test))
    assert 'CustomerID' not in train
    assert 'DurationOfPitch' not in train

def test_saved_model_handles_missing_and_unseen_values():
    model = joblib.load(DEPLOY / 'model.joblib')
    frame = pd.read_csv(SPLITS / 'Xtest.csv').head(2)
    expected = model.predict_proba(frame)
    assert np.allclose(expected.sum(axis=1), 1)
    frame.loc[frame.index[0], 'Age'] = np.nan
    frame.loc[frame.index[1], 'Occupation'] = 'New occupation'
    result = model.predict_proba(frame)
    assert np.isfinite(result).all()
    metadata = json.loads((DEPLOY / 'metadata.json').read_text())
    assert list(model.feature_names_in_) == metadata['features']

def test_app_prediction_and_invalid_party():
    app = AppTest.from_file(str(DEPLOY / 'app.py')).run(timeout=60)
    assert not app.exception
    app.button[0].click().run(timeout=60)
    assert not app.exception
    assert len(app.metric) == 1
    app.number_input[3].set_value(10)
    app.button[0].click().run(timeout=60)
    assert len(app.error) == 1

### Observations

The tests check that identical profiles do not cross the split, the saved pipeline
handles missing and unseen values, and the Streamlit form produces a result. Another
app check confirms that an invalid party size shows an error.

In [ ]:
completed = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'],
                           capture_output=True, text=True, check=False)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError('Project checks failed.')

### Observations

All three local checks passed. The app test actually submits the form and
checks its result. These tests verify local behavior; they do not verify a deployed
Streamlit Community Cloud app.

## 8. MLOps pipeline with GitHub Actions

In [ ]:
%%writefile .github/workflows/pipeline.yml
name: Tourism Project Pipeline

on:
  push:
    branches: [main]
    paths:
      - 'tourism_project/**'
      - 'tests/**'
      - '.github/workflows/pipeline.yml'
  workflow_dispatch:

permissions:
  contents: read
concurrency:
  group: tourism-main
  cancel-in-progress: false
env:
  PYTHON_VERSION: '3.13'

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Register data
        run: python -m tourism_project.model_building.data_register
      - uses: actions/upload-artifact@v4
        with:
          name: registered-data
          path: tourism_project/data/tourism.csv
          if-no-files-found: error
          retention-days: 7

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - run: pip install -r tourism_project/requirements.txt
      - uses: actions/download-artifact@v4
        with:
          name: registered-data
          path: tourism_project/data
      - name: Prepare train and test sets
        run: python -m tourism_project.model_building.prep
      - uses: actions/upload-artifact@v4
        with:
          name: data-splits
          path: tourism_project/splits/*.csv
          if-no-files-found: error
          retention-days: 7

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    permissions:
      contents: write
    steps:
      - uses: actions/checkout@v4
        with:
          ref: main
          fetch-depth: 0
      - name: Check source revision
        run: test "$(git rev-parse HEAD)" = "$GITHUB_SHA"
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - run: pip install -r tourism_project/requirements.txt
      - uses: actions/download-artifact@v4
        with:
          name: data-splits
          path: tourism_project/splits
      - name: Tune and track models
        run: python -m tourism_project.model_building.train
      - name: Test saved model and app
        run: python -m pytest tests -q
      - name: Save experiment records
        uses: actions/upload-artifact@v4
        with:
          name: model-and-experiments
          path: |
            tourism_project/reports/
            tourism_project/deployment/model.joblib
            tourism_project/deployment/metadata.json
            mlruns/
          retention-days: 14
      - name: Commit trained model to main
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add tourism_project/deployment/model.joblib tourism_project/deployment/metadata.json tourism_project/reports/
          if git diff --cached --quiet; then
            echo "No model changes to commit."
          else
            git commit -m "Update tourism model and reports [skip ci]"
            git push origin HEAD:main
          fi

### Observations

The workflow has three dependent jobs: registration, preparation and training.
The preparation job saves four split files as an artifact. The training job downloads
that artifact, tunes the model, tests it and commits the model and reports to main.
Experiment logs are also kept as a downloadable artifact. The automatic token and
`[skip ci]` commit message prevent a retraining loop. A failed test stops the commit.
An outdated source revision stops the run; a normal push rejection also prevents
overwriting a newer main branch. The workflow does not force-push.

In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
.venv/
.ipynb_checkpoints/
mlruns/
tourism_project/splits/
.streamlit/secrets.toml
.env

### Observations

Temporary environments, split files, MLflow working files and secrets are excluded
from normal Git commits. The trained model is deliberately kept in the repository
because the Streamlit app loads it from there.

In [ ]:
import yaml
workflow = yaml.safe_load(Path('.github/workflows/pipeline.yml').read_text())
assert set(workflow['jobs']) == {'register-dataset', 'data-prep', 'model-training'}
assert workflow['jobs']['data-prep']['needs'] == 'register-dataset'
assert workflow['jobs']['model-training']['needs'] == 'data-prep'
required = ['tourism_project/data/tourism.csv',
            'tourism_project/model_building/data_register.py',
            'tourism_project/model_building/prep.py',
            'tourism_project/model_building/train.py',
            'tourism_project/deployment/app.py',
            'tourism_project/deployment/model.joblib',
            'tourism_project/deployment/metadata.json',
            'tourism_project/deployment/requirements.txt',
            'tourism_project/requirements.txt', '.github/workflows/pipeline.yml']
for file in required:
    assert Path(file).is_file(), file
    print('Present:', file)
print('Workflow YAML parsed and job dependencies checked.')

### Observations

The required local files exist, and the YAML can be read with the expected job
dependencies. This is a local structure check. The successful GitHub Actions run linked below also confirms artifact transfer,
model training, app tests and the automatic model commit on GitHub.

## 9. Publish and collect deployment evidence

1. Create a public GitHub repository named `tourism-purchase-prediction` and publish this
   project folder with GitHub Desktop or Git. Include `.github/workflows/pipeline.yml`.
2. Open Actions and run **Tourism Project Pipeline**. Confirm all three jobs finish
   successfully and that a model update is committed to `main`.
3. Sign in to Streamlit Community Cloud. Create an app using the repository, branch
   `main`, and file `tourism_project/deployment/app.py`. Select Python 3.13 in advanced
   settings. Deploy the app and make its access public.
4. Test a prediction in the deployed app. Capture the repository folder structure,
   completed GitHub workflow, and deployed app with their addresses visible.
5. Save the screenshots under `evidence/` and add the real URLs and screenshot paths to
   `deployment_evidence.json`. Rerun the next cell and export this notebook to HTML.

The GitHub workflow updates the committed model when source or data changes. Streamlit
uses the connected repository for app updates. This must be checked after the first cloud run.
Do not put account passwords or tokens in the notebook.

## 10. Output Evaluation

In [ ]:
evidence_path = Path('deployment_evidence.json')
if not evidence_path.exists():
    evidence_path.write_text(json.dumps({key: '' for key in [
        'github_url', 'workflow_url', 'streamlit_url', 'github_screenshot',
        'workflow_screenshot', 'streamlit_screenshot']}, indent=2))
evidence = json.loads(evidence_path.read_text())
for label, key in [('GitHub repository', 'github_url'),
                   ('Executed workflow', 'workflow_url'),
                   ('Streamlit Community Cloud app', 'streamlit_url')]:
    url = evidence.get(key, '')
    if url.startswith('https://'):
        display(Markdown(f'**{label}:** [{url}]({url})'))
    else:
        print(label + ': pending publication and verification')
for label, key in [('GitHub folder structure', 'github_screenshot'),
                   ('Successful workflow', 'workflow_screenshot'),
                   ('Deployed Streamlit app', 'streamlit_screenshot')]:
    filename = evidence.get(key, '')
    if filename and Path(filename).is_file():
        print(label)
        display(Image(filename=filename))
    else:
        print(label + ' screenshot: not yet supplied')
if Path('evidence/streamlit_form.jpg').is_file():
    print('Live app input form')
    display(Image(filename='evidence/streamlit_form.jpg'))

### Observations

The GitHub repository and Streamlit app are public. The linked GitHub run completed
registration, data preparation and model training. The screenshots show the actual
repository, completed workflow and live app. Submitting the default customer form
returned a 4.7% estimated purchase likelihood. This is an example prediction, not
the overall model accuracy. Deployment was checked on 12 October 2026.

## 11. Conclusions and business recommendations

The Random Forest model performs better than the majority baseline at finding buyers. At the training-selected threshold of 0.20, it identifies 143 buyers from a suggested contact list of 261 out of 828 test customers. That gives 54.8% precision and 87.2% recall. There are still 118 unnecessary contacts and 21 missed buyers in this test set. The model is therefore useful for prioritization, but it will make mistakes.

The prediction should support a small marketing pilot. Record which customers were
contacted and which later purchased. Compare conversion rate, contact cost and revenue
with the existing campaign approach before expanding use.

The historical target covers package purchases in general. It cannot directly confirm
interest in Wellness Tourism. The data has no dates, so a future-time test is not possible.
The grouped holdout reduces exact-profile overlap, but does not rule out related or
near-duplicate records. The data collection method and any prior cleaning are unknown.

For a real campaign, confirm that every input is available before the call, check the
designation/product relationship, review sensitive attributes, and choose the threshold
using actual contact capacity and costs. Monitor missing values, category changes and
purchase rates. Retrain when enough new labelled outcomes are available, then review the
new evaluation before using it. The course workflow automatically commits the newly
trained model; a production process should add a business approval and performance gate.

## Rubric coverage

| Rubric item | Evidence in this notebook | Status |
|---|---|---|
| Data registration, 3 points | Validation script, summary and dataset fingerprint | Executed locally |
| Data preparation, 7 points | Cleaning, grouped split, four CSV files, workflow artifact steps | Executed locally and in GitHub Actions |
| Model building and tracking, 13 points | 28 combinations, MLflow runs, test evaluation, saved pipeline | Executed locally |
| Deployment, 11 points | Streamlit app, pinned dependencies, form tests | Live public app tested |
| GitHub Actions, 16 points | Complete three-job YAML and model commit step | Completed successfully on GitHub |
| Output evaluation, 4 points | Evidence section for links and three screenshots | Real URLs and screenshots included |
| Notebook quality, 6 points | Sequential outputs and observations after every code cell | Local notebook executed |

## References

- Supplied Business Context, Rubric, Learner Template Notebook and tourism.csv.
- [scikit-learn grouped stratification](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)
- [GitHub workflow artifacts](https://docs.github.com/en/actions/tutorials/store-and-share-data)
- [GitHub automatic token behavior](https://docs.github.com/en/actions/concepts/security/github_token)
- [Streamlit Community Cloud deployment](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/deploy)
- [MLflow tracking](https://mlflow.org/docs/3.0.1/tracking.html)